Install dependencies

In [1]:
!pip install -q torch transformers accelerate sentencepiece

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


Get SQuAD 2.0 dev set

In [3]:
!wget https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json

--2026-05-02 21:34:22--  https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json
Resolving rajpurkar.github.io (rajpurkar.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to rajpurkar.github.io (rajpurkar.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4370528 (4.2M) [application/json]
Saving to: ‘dev-v2.0.json’

dev-v2.0.json       100%[===================>]   4.17M  --.-KB/s    in 0.06s   

2026-05-02 21:34:22 (72.9 MB/s) - ‘dev-v2.0.json’ saved [4370528/4370528]



Run either pythia or qwen on the SQuAD 2.0 dev set

In [4]:
%%writefile run_squad.py
import json
import os
from typing import List, Dict, Any
import argparse
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEV_FILE = "dev-v2.0.json"
MAX_NEW_TOKENS = 10

# Fixed few-shot examples.
# Use short, clean examples. For unanswerable questions, the answer is "".
FEW_SHOT_EXAMPLES = [
    {
        "context": "The Normans were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France.",
        "question": "In what country is Normandy located?",
        "answer": "France",
    },
    {
        "context": "The Normans were descended from raiders and pirates from Denmark, Iceland and Norway.",
        "question": "From which countries did the Norse originate?",
        "answer": "Denmark, Iceland and Norway",
    },
    {
        "context": "The Normans were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France.",
        "question": "What is France a region of?",
        "answer": "",
    },
]


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_family", type=str, default="pythia", choices=["pythia", "qwen"])
    parser.add_argument("--model_name", type=str, required=True)
    parser.add_argument("--output_file", type=str, required=True)
    parser.add_argument("--max_examples", type=int, default=None)
    parser.add_argument(
        "--prompt_mode",
        type=str,
        default="zero_shot",
        choices=["zero_shot", "few_shot"],
    )
    parser.add_argument("--num_shots", type=int, default=3)
    return parser.parse_args()


def load_squad_v2(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    examples = []
    for article in data["data"]:
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                qid = qa["id"]
                question = qa["question"]
                answers = qa.get("answers", [])
                is_impossible = qa.get("is_impossible", False)

                examples.append(
                    {
                        "id": qid,
                        "context": context,
                        "question": question,
                        "answers": answers,
                        "is_impossible": is_impossible,
                    }
                )
    return examples


def build_qwen_zero_shot_prompt(passage, question):
    return f"""Answer the question using ONLY a short phrase from the passage.

Rules:
- Output only the answer span
- Do not explain
- Do not add extra words
- If not answerable, output: ""

Passage: {passage}
Question: {question}
Answer:"""


def build_qwen_few_shot_prompt(passage, question, num_shots):
    prompt = """Answer the question using ONLY a short phrase from the passage.

Rules:
- Output only the answer span
- Do not explain
- Do not add extra words
- If not answerable, output: ""

"""

    for ex in FEW_SHOT_EXAMPLES[:num_shots]:
        prompt += f"""Passage: {ex['context']}

Question: {ex['question']}
Answer: {ex['answer']}

"""

    prompt += f"""Passage: {passage}
Question: {question}
Answer:"""

    return prompt


def build_pythia_zero_shot_prompt(context: str, question: str) -> str:
    return (
        "Answer using only the shortest phrase from the passage.\n"
        'If not answerable, output: ""\n\n'
        f"Passage: {context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )


def build_pythia_few_shot_prompt(context: str, question: str, num_shots: int) -> str:
    prompt = (
        "Answer each question using only the shortest phrase from the passage.\n"
        'If not answerable, output: ""\n\n'
    )

    for ex in FEW_SHOT_EXAMPLES[:num_shots]:
        prompt += (
            f"Passage: {ex['context']}\n"
            f"Question: {ex['question']}\n"
            f"Answer: {ex['answer']}\n\n"
        )

    prompt += (
        f"Passage: {context}\n"
        f"Question: {question}\n"
        "Answer:"
    )
    return prompt


def build_prompt(context: str, question: str, prompt_mode: str, num_shots: int, model_family: str) -> str:
    if model_family == "qwen":
        if prompt_mode == "zero_shot":
            return build_qwen_zero_shot_prompt(context, question)
        if prompt_mode == "few_shot":
            return build_qwen_few_shot_prompt(context, question, num_shots)

    if model_family == "pythia":
        if prompt_mode == "zero_shot":
            return build_pythia_zero_shot_prompt(context, question)
        if prompt_mode == "few_shot":
            return build_pythia_few_shot_prompt(context, question, num_shots)

    raise ValueError(f"Unknown prompt mode: {prompt_mode}")


def extract_answer(generated_text: str, prompt: str, prompt_mode: str) -> str:
    if generated_text.startswith(prompt):
        answer_text = generated_text[len(prompt):]
    else:
        answer_text = generated_text

    answer_text = answer_text.strip()
    answer = answer_text.split("\n")[0].strip()

    if answer == '""':
        answer = ""
    if not answer:
        answer = ""

    return answer


def generate_prediction(model, tokenizer, prompt: str, device: str, prompt_mode: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    max_new_tokens = 10

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = extract_answer(generated_text, prompt, prompt_mode)
    return answer


def main():
    os.makedirs("predictions", exist_ok=True)

    args = parse_args()
    model_family = args.model_family
    model_name = args.model_name
    output_file = args.output_file
    max_examples = args.max_examples
    prompt_mode = args.prompt_mode
    num_shots = args.num_shots

    print("Loading SQuAD v2.0 dev set...")
    examples = load_squad_v2(DEV_FILE)
    print(f"Loaded {len(examples)} examples.")

    if max_examples is not None:
        examples = examples[:max_examples]
        print(f"Using first {len(examples)} examples for testing.")

    print(f"Prompt mode: {prompt_mode}")
    if prompt_mode == "few_shot":
        print(f"Number of shots: {num_shots}")

    print(f"Loading model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32, device_map="auto")

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    print(f"Using device: {device}")

    predictions = {}

    for i, ex in enumerate(examples, start=1):
        prompt = build_prompt(ex["context"], ex["question"], prompt_mode, num_shots, model_family)
        pred = generate_prediction(model, tokenizer, prompt, device, prompt_mode)
        predictions[ex["id"]] = pred

        if i == 1:
            print("\n=== PROMPT ===\n")
            print(prompt[:1500])
            print("\n=== END PROMPT ===\n")

        if i <= 5:
            print("=" * 80)
            print(f"Example {i}")
            print("Question:", ex["question"])
            print("Gold answers:", [a["text"] for a in ex["answers"]] if ex["answers"] else ['""'])
            print("Prediction:", pred)

        if i % 10 == 0:
            print(f"Processed {i}/{len(examples)} examples...")

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=False)

    print(f"Saved predictions to {output_file}")
    print("Now run the official evaluator:")
    print(f"python evaluate-v2.0.py {DEV_FILE} {output_file}")


if __name__ == "__main__":
    main()


Writing run_squad.py


SQuAD 2.0 Evaluation Script (requires full dev output)

In [5]:
%%writefile evaluate-v2.0.py
"""Official evaluation script for SQuAD version 2.0.

In addition to basic functionality, we also compute additional statistics and
plot precision-recall curves if an additional na_prob.json file is provided.
This file is expected to map question ID's to the model's predicted probability
that a question is unanswerable.
"""
import argparse
import collections
import json
import numpy as np
import os
import re
import string
import sys

OPTS = None

def parse_args():
  parser = argparse.ArgumentParser('Official evaluation script for SQuAD version 2.0.')
  parser.add_argument('data_file', metavar='data.json', help='Input data JSON file.')
  parser.add_argument('pred_file', metavar='pred.json', help='Model predictions.')
  parser.add_argument('--out-file', '-o', metavar='eval.json',
                      help='Write accuracy metrics to file (default is stdout).')
  parser.add_argument('--na-prob-file', '-n', metavar='na_prob.json',
                      help='Model estimates of probability of no answer.')
  parser.add_argument('--na-prob-thresh', '-t', type=float, default=1.0,
                      help='Predict "" if no-answer probability exceeds this (default = 1.0).')
  parser.add_argument('--out-image-dir', '-p', metavar='out_images', default=None,
                      help='Save precision-recall curves to directory.')
  parser.add_argument('--verbose', '-v', action='store_true')
  if len(sys.argv) == 1:
    parser.print_help()
    sys.exit(1)
  return parser.parse_args()

def make_qid_to_has_ans(dataset):
  qid_to_has_ans = {}
  for article in dataset:
    for p in article['paragraphs']:
      for qa in p['qas']:
        qid_to_has_ans[qa['id']] = bool(qa['answers'])
  return qid_to_has_ans

def normalize_answer(s):
  """Lower text and remove punctuation, articles and extra whitespace."""
  def remove_articles(text):
    regex = re.compile(r'\b(a|an|the)\b', re.UNICODE)
    return re.sub(regex, ' ', text)
  def white_space_fix(text):
    return ' '.join(text.split())
  def remove_punc(text):
    exclude = set(string.punctuation)
    return ''.join(ch for ch in text if ch not in exclude)
  def lower(text):
    return text.lower()
  return white_space_fix(remove_articles(remove_punc(lower(s))))

def get_tokens(s):
  if not s: return []
  return normalize_answer(s).split()

def compute_exact(a_gold, a_pred):
  return int(normalize_answer(a_gold) == normalize_answer(a_pred))

def compute_f1(a_gold, a_pred):
  gold_toks = get_tokens(a_gold)
  pred_toks = get_tokens(a_pred)
  common = collections.Counter(gold_toks) & collections.Counter(pred_toks)
  num_same = sum(common.values())
  if len(gold_toks) == 0 or len(pred_toks) == 0:
    # If either is no-answer, then F1 is 1 if they agree, 0 otherwise
    return int(gold_toks == pred_toks)
  if num_same == 0:
    return 0
  precision = 1.0 * num_same / len(pred_toks)
  recall = 1.0 * num_same / len(gold_toks)
  f1 = (2 * precision * recall) / (precision + recall)
  return f1

def get_raw_scores(dataset, preds):
  exact_scores = {}
  f1_scores = {}
  for article in dataset:
    for p in article['paragraphs']:
      for qa in p['qas']:
        qid = qa['id']
        gold_answers = [a['text'] for a in qa['answers']
                        if normalize_answer(a['text'])]
        if not gold_answers:
          # For unanswerable questions, only correct answer is empty string
          gold_answers = ['']
        if qid not in preds:
          print('Missing prediction for %s' % qid)
          continue
        a_pred = preds[qid]
        # Take max over all gold answers
        exact_scores[qid] = max(compute_exact(a, a_pred) for a in gold_answers)
        f1_scores[qid] = max(compute_f1(a, a_pred) for a in gold_answers)
  return exact_scores, f1_scores

def apply_no_ans_threshold(scores, na_probs, qid_to_has_ans, na_prob_thresh):
  new_scores = {}
  for qid, s in scores.items():
    pred_na = na_probs[qid] > na_prob_thresh
    if pred_na:
      new_scores[qid] = float(not qid_to_has_ans[qid])
    else:
      new_scores[qid] = s
  return new_scores

def make_eval_dict(exact_scores, f1_scores, qid_list=None):
  if not qid_list:
    total = len(exact_scores)
    return collections.OrderedDict([
        ('exact', 100.0 * sum(exact_scores.values()) / total),
        ('f1', 100.0 * sum(f1_scores.values()) / total),
        ('total', total),
    ])
  else:
    total = len(qid_list)
    return collections.OrderedDict([
        ('exact', 100.0 * sum(exact_scores[k] for k in qid_list) / total),
        ('f1', 100.0 * sum(f1_scores[k] for k in qid_list) / total),
        ('total', total),
    ])

def merge_eval(main_eval, new_eval, prefix):
  for k in new_eval:
    main_eval['%s_%s' % (prefix, k)] = new_eval[k]

def plot_pr_curve(precisions, recalls, out_image, title):
  plt.step(recalls, precisions, color='b', alpha=0.2, where='post')
  plt.fill_between(recalls, precisions, step='post', alpha=0.2, color='b')
  plt.xlabel('Recall')
  plt.ylabel('Precision')
  plt.xlim([0.0, 1.05])
  plt.ylim([0.0, 1.05])
  plt.title(title)
  plt.savefig(out_image)
  plt.clf()

def make_precision_recall_eval(scores, na_probs, num_true_pos, qid_to_has_ans,
                               out_image=None, title=None):
  qid_list = sorted(na_probs, key=lambda k: na_probs[k])
  true_pos = 0.0
  cur_p = 1.0
  cur_r = 0.0
  precisions = [1.0]
  recalls = [0.0]
  avg_prec = 0.0
  for i, qid in enumerate(qid_list):
    if qid_to_has_ans[qid]:
      true_pos += scores[qid]
    cur_p = true_pos / float(i+1)
    cur_r = true_pos / float(num_true_pos)
    if i == len(qid_list) - 1 or na_probs[qid] != na_probs[qid_list[i+1]]:
      # i.e., if we can put a threshold after this point
      avg_prec += cur_p * (cur_r - recalls[-1])
      precisions.append(cur_p)
      recalls.append(cur_r)
  if out_image:
    plot_pr_curve(precisions, recalls, out_image, title)
  return {'ap': 100.0 * avg_prec}

def run_precision_recall_analysis(main_eval, exact_raw, f1_raw, na_probs,
                                  qid_to_has_ans, out_image_dir):
  if out_image_dir and not os.path.exists(out_image_dir):
    os.makedirs(out_image_dir)
  num_true_pos = sum(1 for v in qid_to_has_ans.values() if v)
  if num_true_pos == 0:
    return
  pr_exact = make_precision_recall_eval(
      exact_raw, na_probs, num_true_pos, qid_to_has_ans,
      out_image=os.path.join(out_image_dir, 'pr_exact.png'),
      title='Precision-Recall curve for Exact Match score')
  pr_f1 = make_precision_recall_eval(
      f1_raw, na_probs, num_true_pos, qid_to_has_ans,
      out_image=os.path.join(out_image_dir, 'pr_f1.png'),
      title='Precision-Recall curve for F1 score')
  oracle_scores = {k: float(v) for k, v in qid_to_has_ans.items()}
  pr_oracle = make_precision_recall_eval(
      oracle_scores, na_probs, num_true_pos, qid_to_has_ans,
      out_image=os.path.join(out_image_dir, 'pr_oracle.png'),
      title='Oracle Precision-Recall curve (binary task of HasAns vs. NoAns)')
  merge_eval(main_eval, pr_exact, 'pr_exact')
  merge_eval(main_eval, pr_f1, 'pr_f1')
  merge_eval(main_eval, pr_oracle, 'pr_oracle')

def histogram_na_prob(na_probs, qid_list, image_dir, name):
  if not qid_list:
    return
  x = [na_probs[k] for k in qid_list]
  weights = np.ones_like(x) / float(len(x))
  plt.hist(x, weights=weights, bins=20, range=(0.0, 1.0))
  plt.xlabel('Model probability of no-answer')
  plt.ylabel('Proportion of dataset')
  plt.title('Histogram of no-answer probability: %s' % name)
  plt.savefig(os.path.join(image_dir, 'na_prob_hist_%s.png' % name))
  plt.clf()

def find_best_thresh(preds, scores, na_probs, qid_to_has_ans):
  num_no_ans = sum(1 for k in qid_to_has_ans if not qid_to_has_ans[k])
  cur_score = num_no_ans
  best_score = cur_score
  best_thresh = 0.0
  qid_list = sorted(na_probs, key=lambda k: na_probs[k])
  for i, qid in enumerate(qid_list):
    if qid not in scores: continue
    if qid_to_has_ans[qid]:
      diff = scores[qid]
    else:
      if preds[qid]:
        diff = -1
      else:
        diff = 0
    cur_score += diff
    if cur_score > best_score:
      best_score = cur_score
      best_thresh = na_probs[qid]
  return 100.0 * best_score / len(scores), best_thresh

def find_all_best_thresh(main_eval, preds, exact_raw, f1_raw, na_probs, qid_to_has_ans):
  best_exact, exact_thresh = find_best_thresh(preds, exact_raw, na_probs, qid_to_has_ans)
  best_f1, f1_thresh = find_best_thresh(preds, f1_raw, na_probs, qid_to_has_ans)
  main_eval['best_exact'] = best_exact
  main_eval['best_exact_thresh'] = exact_thresh
  main_eval['best_f1'] = best_f1
  main_eval['best_f1_thresh'] = f1_thresh

def main():
  with open(OPTS.data_file) as f:
    dataset_json = json.load(f)
    dataset = dataset_json['data']
  with open(OPTS.pred_file) as f:
    preds = json.load(f)
  if OPTS.na_prob_file:
    with open(OPTS.na_prob_file) as f:
      na_probs = json.load(f)
  else:
    na_probs = {k: 0.0 for k in preds}
  qid_to_has_ans = make_qid_to_has_ans(dataset)  # maps qid to True/False
  has_ans_qids = [k for k, v in qid_to_has_ans.items() if v]
  no_ans_qids = [k for k, v in qid_to_has_ans.items() if not v]
  exact_raw, f1_raw = get_raw_scores(dataset, preds)
  exact_thresh = apply_no_ans_threshold(exact_raw, na_probs, qid_to_has_ans,
                                        OPTS.na_prob_thresh)
  f1_thresh = apply_no_ans_threshold(f1_raw, na_probs, qid_to_has_ans,
                                     OPTS.na_prob_thresh)
  out_eval = make_eval_dict(exact_thresh, f1_thresh)
  if has_ans_qids:
    has_ans_eval = make_eval_dict(exact_thresh, f1_thresh, qid_list=has_ans_qids)
    merge_eval(out_eval, has_ans_eval, 'HasAns')
  if no_ans_qids:
    no_ans_eval = make_eval_dict(exact_thresh, f1_thresh, qid_list=no_ans_qids)
    merge_eval(out_eval, no_ans_eval, 'NoAns')
  if OPTS.na_prob_file:
    find_all_best_thresh(out_eval, preds, exact_raw, f1_raw, na_probs, qid_to_has_ans)
  if OPTS.na_prob_file and OPTS.out_image_dir:
    run_precision_recall_analysis(out_eval, exact_raw, f1_raw, na_probs,
                                  qid_to_has_ans, OPTS.out_image_dir)
    histogram_na_prob(na_probs, has_ans_qids, OPTS.out_image_dir, 'hasAns')
    histogram_na_prob(na_probs, no_ans_qids, OPTS.out_image_dir, 'noAns')
  if OPTS.out_file:
    with open(OPTS.out_file, 'w') as f:
      json.dump(out_eval, f)
  else:
    print(json.dumps(out_eval, indent=2))

if __name__ == '__main__':
  OPTS = parse_args()
  if OPTS.out_image_dir:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
  main()


Writing evaluate-v2.0.py


In [6]:
!python run_squad.py \
  --model_family pythia \
  --model_name EleutherAI/pythia-70m \
  --output_file predictions_pythia_70m_few.json \
  --prompt_mode few_shot \
  --num_shots 3

Loading SQuAD v2.0 dev set...
Loaded 11873 examples.
Prompt mode: few_shot
Number of shots: 3
Loading model: EleutherAI/pythia-70m
config.json: 100% 567/567 [00:00<00:00, 2.11MB/s]
tokenizer_config.json: 100% 396/396 [00:00<00:00, 1.77MB/s]
tokenizer.json: 2.11MB [00:00, 14.0MB/s]
special_tokens_map.json: 100% 99.0/99.0 [00:00<00:00, 483kB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 166M/166M [00:04<00:00, 36.8MB/s]
Loading weights: 100% 76/76 [00:00<00:00, 1401.25it/s, Materializing param=gpt_neox.layers.5.post_attention_layernorm.weight]
Using device: cuda

=== PROMPT ===

Answer each question using only the shortest phrase from the passage.
If not answerable, output: ""

Passage: The Normans were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France.
Question: In what country is Normandy located?
Answer: France

Passage: The Normans were descended from raiders and pirates from Denmark, Iceland and Norway.
Quest

In [6]:
!python evaluate-v2.0.py dev-v2.0.json predictions_pythia_70m_few.json

{
  "exact": 0.042112355765181506,
  "f1": 2.9828060069921922,
  "total": 11873,
  "HasAns_exact": 0.08434547908232119,
  "HasAns_f1": 5.974165944841143,
  "HasAns_total": 5928,
  "NoAns_exact": 0.0,
  "NoAns_f1": 0.0,
  "NoAns_total": 5945
}


In [ ]:
!python run_squad.py \
  --model_family qwen \
  --model_name Qwen/Qwen2.5-3B-Instruct \
  --output_file predictions/predictions_qwen2.5_3b_few.json \
  --prompt_mode few_shot \
  --num_shots 3

Loading SQuAD v2.0 dev set...
Loaded 11873 examples.
Prompt mode: few_shot
Number of shots: 3
Loading model: Qwen/Qwen2.5-3B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:08<00:00, 51.12it/s, Materializing param=model.norm.weight]
Using device: cuda
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

=== PROMPT ===

Answer the question using ONLY a short phrase from the passage.

Rules:
- Output only the answer span
- Do not explain
- Do not add extra words
- If not answerable, output: ""

Passage: The Normans were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France.

Question: In what country is Normandy located?
Answer: France

Passage: The Normans were descended from raiders and pirates from Denmark, Iceland and Norway.

Question: From which countries did the Norse originate?
Answer: Denmark, Ice